# Assignment 2: The Dirty Ledger - cleaning on a copy
Note: the brief names `teaching_prices_messy.csv`, but the file supplied is `teaching_prices_clean.csv`. This notebook audits the supplied file. Nothing is changed in the original; all work is on a copy.

## Part A. Load and keep a copy

In [1]:
import pandas as pd

raw = pd.read_csv('teaching_prices_clean.csv')
df = raw.copy()   # all work happens on df, raw is never touched
print(raw.shape)

(96, 6)


## Part B. BEFORE table (diagnosis checks)

In [2]:
print('rows, columns:', df.shape)
print()
print(df.dtypes)
print()
print('missing per column:')
print(df.isna().sum())
print()
print('fully duplicated rows:', df.duplicated().sum())

rows, columns: (96, 6)

date              str
item              str
city              str
price         float64
unit              str
data_label        str
dtype: object

missing per column:
date          0
item          0
city          0
price         0
unit          0
data_label    0
dtype: int64

fully duplicated rows: 0


In [3]:
print(df.describe())
print()
print(df['price'].describe())

            price
count   96.000000
mean   202.166667
std     79.374949
min     99.500000
25%    142.250000
50%    192.250000
75%    234.625000
max    350.000000

count     96.000000
mean     202.166667
std       79.374949
min       99.500000
25%      142.250000
50%      192.250000
75%      234.625000
max      350.000000
Name: price, dtype: float64


In [4]:
for col in ['item', 'city', 'unit', 'data_label']:
    print(df[col].value_counts())
    print()

item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
Name: count, dtype: int64

city
City 1    48
City 2    48
Name: count, dtype: int64

unit
PKR per kg (illustrative)    96
Name: count, dtype: int64

data_label
ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS    96
Name: count, dtype: int64



In [5]:
print('dates and rows per date:')
print(df['date'].value_counts().sort_index())

dates and rows per date:
date
2026-01-01    8
2026-01-08    8
2026-01-15    8
2026-01-22    8
2026-01-29    8
2026-02-05    8
2026-02-12    8
2026-02-19    8
2026-02-26    8
2026-03-05    8
2026-03-12    8
2026-03-19    8
Name: count, dtype: int64


In [6]:
# Do labels have stray spaces or odd case?
for col in ['item', 'city', 'unit', 'date']:
    print(col, 'with spaces at the ends:', (df[col] != df[col].str.strip()).sum())

item with spaces at the ends: 0
city with spaces at the ends: 0
unit with spaces at the ends: 0
date with spaces at the ends: 0


In [7]:
# Are dates one week apart?
d = pd.to_datetime(df['date']).drop_duplicates().sort_values()
print(d.diff().value_counts())

date
7 days    11
Name: count, dtype: int64


In [8]:
# Is City 2 always City 1 + same amount?
wide = df.pivot_table(index=['date', 'item'], columns='city', values='price')
print((wide['City 2'] - wide['City 1']).value_counts())

6.0    48
Name: count, dtype: int64


In [9]:
# Biggest week-to-week price change per item and city
tmp = df.sort_values('date').copy()
tmp['change_pct'] = tmp.groupby(['item', 'city'])['price'].pct_change() * 100
print(tmp['change_pct'].abs().max())
print(tmp.loc[tmp['change_pct'].abs().idxmax()])

3.5175879396984966
date                                                 2026-01-15
item                                                   Staple A
city                                                     City 1
price                                                     103.0
unit                                  PKR per kg (illustrative)
data_label    ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
change_pct                                             3.517588
Name: 16, dtype: object


In [10]:
print('prices <= 0:', (df['price'] <= 0).sum())
print('prices with .5:', (df['price'] % 1 != 0).sum())
print()
print(df.groupby('item')['price'].agg(['count', 'min', 'max', 'mean']))

prices <= 0: 0
prices with .5: 36

             count    min    max        mean
item                                        
Oil B           24  298.0  350.0  324.833333
Pulse C         24  200.0  213.5  205.916667
Staple A        24   99.5  122.0  111.166667
Sweetener D     24  149.0  184.5  166.750000


In [11]:
before = pd.DataFrame({
    'rows': [len(df)],
    'missing_total': [df.isna().sum().sum()],
    'duplicate_rows': [df.duplicated().sum()],
    'price_mean': [df['price'].mean()],
    'price_min': [df['price'].min()],
    'price_max': [df['price'].max()],
}, index=['BEFORE'])
before

,rows,missing_total,duplicate_rows,price_mean,price_min,price_max
BEFORE,96,0,0,202.166667,99.5,350.0


## Part C. The Cleaning Code Court
The AI-suggested cleaning code is below, one line at a time. Each line is tried on its own fresh copy (`t`) so a bad line cannot damage the real data. I print rows, missing count and any changes for each.

### Line 1: `df.columns = df.columns.str.strip().str.lower()`

In [12]:
t = raw.copy()
t.columns = t.columns.str.strip().str.lower()
print('rows', len(t), '| columns same as before:', list(t.columns) == list(raw.columns))

rows 96 | columns same as before: True


### Line 2: `df['item'] = df['item'].str.strip().str.lower()`

In [13]:
t = raw.copy()
t['item'] = t['item'].str.strip().str.lower()
print('rows', len(t))
print('labels before:', sorted(raw['item'].unique()))
print('labels after: ', sorted(t['item'].unique()))
print('labels changed:', (t['item'] != raw['item']).sum())

rows 96
labels before: ['Oil B', 'Pulse C', 'Staple A', 'Sweetener D']
labels after:  ['oil b', 'pulse c', 'staple a', 'sweetener d']
labels changed: 96


In [14]:
# Modified version: strip only, keep the original capitalisation
t = raw.copy()
t['item'] = t['item'].str.strip()
print('labels changed:', (t['item'] != raw['item']).sum())

labels changed: 0


### Line 3: `df['date'] = pd.to_datetime(df['date'])`

In [15]:
t = raw.copy()
t['date'] = pd.to_datetime(t['date'])
print(t['date'].dtype)
print('rows', len(t), '| missing dates:', t['date'].isna().sum())
print('first / last:', t['date'].min(), t['date'].max())

datetime64[us]
rows 96 | missing dates: 0
first / last: 2026-01-01 00:00:00 2026-03-19 00:00:00


### Line 4: `df['price'] = pd.to_numeric(df['price'], errors='coerce')`

In [16]:
t = raw.copy()
t['price'] = pd.to_numeric(t['price'], errors='coerce')
print('missing prices created:', t['price'].isna().sum())
print('prices changed:', (t['price'] != raw['price']).sum())

missing prices created: 0
prices changed: 0


In [17]:
# Modified version: errors='raise' so a bad value stops the code instead of silently becoming NaN
t = raw.copy()
t['price'] = pd.to_numeric(t['price'], errors='raise')
print('prices changed:', (t['price'] != raw['price']).sum())

prices changed: 0


### Line 5: `df = df.dropna()`

In [18]:
t = raw.copy()
print('missing before:', t.isna().sum().sum())
t = t.dropna()
print('rows before', len(raw), '| rows after', len(t), '| removed', len(raw) - len(t))

missing before: 0
rows before 96 | rows after 96 | removed 0


### Line 6: `df = df.drop_duplicates(subset=['date', 'item'])`

In [19]:
t = raw.copy()
t = t.drop_duplicates(subset=['date', 'item'])
print('rows before', len(raw), '| rows after', len(t), '| removed', len(raw) - len(t))
print()
print(t['city'].value_counts())

rows before 96 | rows after 48 | removed 48

city
City 1    48
Name: count, dtype: int64


In [20]:
# Which rows did it remove?
removed = raw[~raw.index.isin(t.index)]
print(removed['city'].value_counts())
print(removed.head())

city
City 2    48
Name: count, dtype: int64
         date         item    city  price                       unit  \
1  2026-01-01     Staple A  City 2  107.0  PKR per kg (illustrative)   
3  2026-01-01        Oil B  City 2  304.0  PKR per kg (illustrative)   
5  2026-01-01      Pulse C  City 2  206.0  PKR per kg (illustrative)   
7  2026-01-01  Sweetener D  City 2  155.0  PKR per kg (illustrative)   
9  2026-01-08     Staple A  City 2  105.5  PKR per kg (illustrative)   

                                          data_label  
1  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
3  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
5  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
7  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
9  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  


In [21]:
# Check for duplicates the correct way: date + item + city together
print('duplicates on date+item+city:', raw.duplicated(subset=['date', 'item', 'city']).sum())
print('fully duplicated rows:', raw.duplicated().sum())

duplicates on date+item+city: 0
fully duplicated rows: 0


### Line 7: `df['price'] = df['price'].fillna(df['price'].median())`

In [22]:
t = raw.copy()
print('median that would be used:', t['price'].median())
t['price'] = t['price'].fillna(t['price'].median())
print('prices changed:', (t['price'] != raw['price']).sum())
print('missing prices to fill:', raw['price'].isna().sum())

median that would be used: 192.25
prices changed: 0
missing prices to fill: 0


### Line 8: outlier removal with the IQR rule on the whole price column

In [23]:
t = raw.copy()
q1 = t['price'].quantile(0.25)
q3 = t['price'].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print('fences:', low, high)
t = t[(t['price'] >= low) & (t['price'] <= high)]
print('rows before', len(raw), '| rows after', len(t), '| removed', len(raw) - len(t))

fences: 3.6875 373.1875
rows before 96 | rows after 96 | removed 0


In [24]:
# The same rule, but inside each item (a fairer test)
t = raw.copy()
keep = []
for name, g in t.groupby('item'):
    q1 = g['price'].quantile(0.25)
    q3 = g['price'].quantile(0.75)
    iqr = q3 - q1
    ok = (g['price'] >= q1 - 1.5 * iqr) & (g['price'] <= q3 + 1.5 * iqr)
    keep.append(g[ok])
t = pd.concat(keep)
print('rows before', len(raw), '| rows after', len(t), '| removed', len(raw) - len(t))

rows before 96 | rows after 96 | removed 0


### Line 9: `df['price'] = df['price'].round(0)`

In [25]:
t = raw.copy()
t['price'] = t['price'].round(0)
print('prices changed:', (t['price'] != raw['price']).sum())
changed = t[t['price'] != raw['price']]
print(pd.DataFrame({'before': raw.loc[changed.index, 'price'], 'after': changed['price']}).head(10))

prices changed: 36
    before  after
8     99.5  100.0
9    105.5  106.0
12   202.5  202.0
13   208.5  208.0
14   153.5  154.0
15   159.5  160.0
24   106.5  106.0
25   112.5  112.0
28   202.5  202.0
29   208.5  208.0


### Line 10: `df = df.drop(columns=['data_label'])`

In [26]:
t = raw.copy()
print('data_label values:', raw['data_label'].unique())
t = t.drop(columns=['data_label'])
print('columns before:', list(raw.columns))
print('columns after: ', list(t.columns))

data_label values:

 <StringArray>
['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']
Length: 1, dtype: str
columns before: ['date', 'item', 'city', 'price', 'unit', 'data_label']
columns after:  ['date', 'item', 'city', 'price', 'unit']


### Line 11: `df = df.sort_values(['date', 'item', 'city']).reset_index(drop=True)`

In [27]:
t = raw.copy()
t = t.sort_values(['date', 'item', 'city']).reset_index(drop=True)
print('rows', len(t))
# same rows, just in a different order?
a = raw.sort_values(['date', 'item', 'city']).reset_index(drop=True)
print('identical content after sorting both:', a.equals(t))

rows 96
identical content after sorting both: True


## Part D. Final cleaning (only the kept and modified lines)

In [28]:
clean = raw.copy()

# L1 keep
clean.columns = clean.columns.str.strip().str.lower()

# L2 modified: strip only
clean['item'] = clean['item'].str.strip()
clean['city'] = clean['city'].str.strip()

# L3 keep
clean['date'] = pd.to_datetime(clean['date'])

# L4 modified: raise instead of coerce
clean['price'] = pd.to_numeric(clean['price'], errors='raise')

# L5 modified: report first, drop only rows with a missing price
print('rows with missing price:', clean['price'].isna().sum())
clean = clean.dropna(subset=['price'])

# L11 keep
clean = clean.sort_values(['date', 'item', 'city']).reset_index(drop=True)

print('final rows:', len(clean))

rows with missing price:

 0
final rows: 96


## Part E. AFTER table and checks

In [29]:
after = pd.DataFrame({
    'rows': [len(clean)],
    'missing_total': [clean.isna().sum().sum()],
    'duplicate_rows': [clean.duplicated().sum()],
    'price_mean': [clean['price'].mean()],
    'price_min': [clean['price'].min()],
    'price_max': [clean['price'].max()],
}, index=['AFTER'])
print(pd.concat([before, after]))

        rows  missing_total  duplicate_rows  price_mean  price_min  price_max
BEFORE    96              0               0  202.166667       99.5      350.0
AFTER     96              0               0  202.166667       99.5      350.0

In [30]:
print(clean.describe())
print()
print(clean.dtypes)

                      date       price
count                   96   96.000000
mean   2026-02-08 12:00:00  202.166667
min    2026-01-01 00:00:00   99.500000
25%    2026-01-20 06:00:00  142.250000
50%    2026-02-08 12:00:00  192.250000
75%    2026-02-27 18:00:00  234.625000
max    2026-03-19 00:00:00  350.000000
std                    NaN   79.374949



date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str
dtype: object


In [31]:
# Was any valid observation removed? Compare every row to the original.
a = raw.sort_values(['date', 'item', 'city']).reset_index(drop=True)
b = clean.copy()
b['date'] = b['date'].dt.strftime('%Y-%m-%d')
print('same rows, same values, nothing changed except date type:', a.equals(b))
print('prices changed:', (a['price'] != b['price']).sum())
print('labels changed:', (a['item'] != b['item']).sum() + (a['city'] != b['city']).sum())

same rows, same values, nothing changed except date type: True
prices changed: 0
labels changed: 0


In [32]:
# Does every item appear the number of times I expect?
# Expected: 12 dates x 2 cities = 24 per item, and 8 rows per date (4 items x 2 cities)
print(clean['item'].value_counts())
print()
print(clean['city'].value_counts())
print()
print('rows per date, unique counts:', clean['date'].value_counts().unique())
print('every item has 24 rows:', (clean['item'].value_counts() == 24).all())
print('every date has 8 rows:', (clean['date'].value_counts() == 8).all())

item
Oil B          24
Pulse C        24
Staple A       24
Sweetener D    24
Name: count, dtype: int64

city
City 1    48
City 2    48
Name: count, dtype: int64

rows per date, unique counts: [8]
every item has 24 rows: True


every date has 8 rows: True


In [33]:
# Save the cleaned copy (original file is untouched)
clean.to_csv('teaching_prices_cleaned_output.csv', index=False)
print('saved')

saved
